# Notebook 01 — Why Is My Monthly Report Instant But My Colleague's Takes Forever?
## The Hidden Power of Natural Data Ordering

**The scenario**: You and a colleague both query the same 2 billion transactions. Your monthly report runs in seconds. Theirs takes 10x longer. Same query, same data, same warehouse — what gives?

**The answer**: Snowflake stores data in micro-partitions. When data is loaded in date order, date-range queries only scan the relevant partitions. When data is shuffled, Snowflake must scan everything.

| Table | How It Was Loaded | Expected Behavior |
|-------|-------------------|-------------------|
| `TRANSACTIONS` | Sequential date batches | Date queries prune ~90% of partitions |
| `TRANSACTIONS_UNORDERED` | `ORDER BY RANDOM()` | Date queries scan ALL partitions |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Best Case: Date-Ordered Table

"Give me November 2024 transaction totals by channel."

In [ ]:
-- BEST: Naturally ordered table — only scans Nov 2024 partitions
SELECT
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_volume
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1
ORDER BY total_volume DESC;

---
## Worst Case: Randomly Shuffled Table

Same exact query, same data — but the table was loaded in random order.

In [ ]:
-- WORST: Randomly shuffled — must scan ALL partitions to find Nov 2024
SELECT
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_volume
FROM TRANSACTIONS_UNORDERED
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1
ORDER BY total_volume DESC;

---
## Compare the Results

In [ ]:
-- Compare bytes scanned and time
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_UNORDERED%' THEN 'UNORDERED (worst)'
        ELSE 'ORDERED (best)'
    END AS approach,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_WAREHOUSE(
    WAREHOUSE_NAME => 'HOL_XS',
    END_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%2024-11-01%'
  AND query_text ILIKE '%channel%'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Key Takeaway

| Metric | Ordered Table | Unordered Table | Difference |
|--------|--------------|-----------------|------------|
| GB Scanned | ~1-2 GB | ~18+ GB | **10-15x more** |
| Time | ~2-3 sec | ~20-30 sec | **10x slower** |

**Why it matters for you as an analyst:**
- Tables loaded by ETL pipelines in date order give you free performance on date-range queries
- If your query is slow on a date filter, check with your data engineer — the table might have been bulk-loaded out of order
- You can't fix this yourself (it requires reloading data), but knowing WHY helps you file the right ticket

**Next** → [Notebook 02 — Clustering](./Notebook_02_Clustering.ipynb) (what to do when you filter on non-date columns)